In [ ]:
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# 1. 加载纽约数据集
print("================ 1. 加载纽约精细栅格数据集 ================")
ny_file = r"E:\chuli\NY_cleaned_dataset.csv"
df = pd.read_csv(ny_file)

# 2. 打印数据集维度
print(f"✅ 数据集加载成功！共包含 {df.shape[0]} 个栅格 (行), {df.shape[1]} 个特征字段 (列)。\n")

# 3. 完整打印所有列名 (带序号，方便查找)
print("================ 2. 完整字段列表 (Columns) ================")
columns_list = df.columns.tolist()
for i, col in enumerate(columns_list):
    print(f"{i+1:03d} | {col}")

# 4. 为了让你看清楚犯罪数据的具体长相，我们筛选出带有 "Year" 或者 "Freq" 等关键字段打印一下
print("\n================ 3. 疑似目标变量预览 (前 5 行) ================")
# 智能抓取可能代表目标变量的列
target_candidates = [col for col in columns_list if 'year' in col.lower() or 'freq' in col.lower() or 'crime' in col.lower()]
if target_candidates:
    print(df[target_candidates].head())
else:
    print("未自动检测到带 'year' 或 'freq' 的列，显示前 10 列预览：")
    print(df.iloc[:, :10].head())

In [ ]:
import os
import numpy as np
import pandas as pd
import rasterio
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import warnings

warnings.filterwarnings('ignore')

print("================ 1. 从对齐栅格正确提取数据 (保留安全网格) ================")
folder_path = r"E:\chuli\NY_aligned"
raster_files = [f for f in os.listdir(folder_path) if f.endswith(('.tif', '.tiff'))]

# 读取坐标系
ref_file = os.path.join(folder_path, raster_files[0])
with rasterio.open(ref_file) as src:
    width, height = src.width, src.height
    transform = src.transform
    cols, rows = np.meshgrid(np.arange(width), np.arange(height))
    xs, ys = rasterio.transform.xy(transform, rows, cols, offset='center')

data_dict = {
    'POINT_X': np.array(xs).flatten(),
    'POINT_Y': np.array(ys).flatten()
}

# 提取所有波段像素
for file_name in raster_files:
    col_name = os.path.splitext(file_name)[0]
    with rasterio.open(os.path.join(folder_path, file_name)) as src:
        data_dict[col_name] = src.read(1).flatten()

df = pd.DataFrame(data_dict)

# =========================================================
# 2. 核心修复：以“路网”为城市边界掩膜，保留0值格子并赋值！
# =========================================================
print("\n================ 2. 执行空间掩膜与 0 值强制填充 ================")
# 我们选择路网特征作为绝对的城市陆地边界 (只要有路网的地方，即使没犯罪也是陆地)
env_mask_col = 'NY_road_cut_raster_clipped' 

# 找到路网的 NoData 值，剔除真正的海洋/边界外区域
with rasterio.open(os.path.join(folder_path, env_mask_col + '.tif')) as src:
    env_nodata = src.nodata

if env_nodata is not None:
    df_clean = df[df[env_mask_col] != env_nodata].copy()
else:
    # 兜底：过滤掉极端的负数背景值 (-9999 或 -3.4e38)
    df_clean = df[df[env_mask_col] > -999].copy()

print(f"🌍 划定纽约有效陆地范围：{len(df_clean)} 个基础微观格子。")

# 🌟 听从指令：给所有犯罪列的空值/NoData 强制赋值为 0
crime_and_year_cols = [col for col in df_clean.columns if 'CrimeType' in col or 'Year_' in col]
for col in crime_and_year_cols:
    with rasterio.open(os.path.join(folder_path, col + '.tif')) as src:
        c_nodata = src.nodata
    if c_nodata is not None:
        df_clean[col] = df_clean[col].replace(c_nodata, 0)
    
    # 强制将所有负数背景值（如 -9999）和 NaN 统统变为 0 (这就是安全的格子！)
    df_clean[col] = np.where(df_clean[col] < 0, 0, df_clean[col])
    df_clean[col] = df_clean[col].fillna(0)

print("🛡️ 已将所有未发生犯罪的区域强制赋值为 0，成功保留安全样本！")

# =========================================================
# 3. 生成 100m 粗栅格并聚合
# =========================================================
GRID_SIZE = 0.001  # 约 100米
print(f"\n================ 3. 生成 100m 粗栅格 (GRID_SIZE={GRID_SIZE}) ================")
df_clean['coarse_x'] = (df_clean['POINT_X'] // GRID_SIZE) * GRID_SIZE
df_clean['coarse_y'] = (df_clean['POINT_Y'] // GRID_SIZE) * GRID_SIZE

agg_rules = {}
for col in df_clean.columns:
    if col in ['coarse_x', 'coarse_y', 'POINT_X', 'POINT_Y']: continue
    if 'freq' in col.lower() or 'year_' in col.lower() or 'total' in col.lower():
        agg_rules[col] = 'sum'
    else:
        agg_rules[col] = 'mean'

df_coarse = df_clean.groupby(['coarse_x', 'coarse_y']).agg(agg_rules).reset_index()
print(f"✅ 合并完成！生成 {len(df_coarse)} 个 100m 粗栅格单元。")

# =========================================================
# 4. 纽约五大犯罪纯环境定性评估 (包含 0 值的二分类)
# =========================================================
print("\n================ 4. 执行三大模型纯环境评估 ================")
env_features = [col for col in df_coarse.columns if 
                (any(k in col for k in ['ratio_', 'accessibility', 'density', 'road', 'landscan'])) and 
                ('arrest' not in col.lower())]

crime_targets = {
    'Drug and traffic': 'CrimeType_Drug_and_traffic_crimes_Freq',
    'Property': 'CrimeType_Property_crimes_Freq',
    'Public order': 'CrimeType_Public_order_and_other_crimes_Freq',
    'Sexual/Children': 'CrimeType_Sexual_crimes_and_crimes_against_children_Freq',
    'Violent': 'CrimeType_Violent_crimes_Freq'
}

all_results = []
for crime_name, crime_col in crime_targets.items():
    print(f"🚀 正在分析: 【{crime_name}】 ...")
    
    # 这一次，y_binary 中会有大量完美的 0 (安全区域) 和 1 (危险区域)
    y_binary = (df_coarse[crime_col] > 0).astype(int)
    
    X = df_coarse[env_features]
    y = y_binary
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    models = {
        "XGBoost": xgb.XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42, n_jobs=-1, eval_metric='logloss'),
        "LightGBM": lgb.LGBMClassifier(n_estimators=200, num_leaves=31, learning_rate=0.05, random_state=42, n_jobs=-1, verbose=-1),
        "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
    }
    
    for model_name, clf in models.items():
        clf.fit(X_train, y_train)
        prob_test = clf.predict_proba(X_test)[:, 1]
        pred_test = clf.predict(X_test)
        
        auc = roc_auc_score(y_test, prob_test) if len(np.unique(y_test)) > 1 else np.nan
        rec = recall_score(y_test, pred_test, zero_division=0)
        f1 = f1_score(y_test, pred_test, zero_division=0)
        acc = accuracy_score(y_test, pred_test)
        
        all_results.append({'Crime Type': crime_name, 'Model': model_name, 'Cls_AUC': auc, 'Recall': rec, 'F1_Score': f1, 'Accuracy': acc})

print("\n================ 🏆 纽约 100m 粗栅格：纯环境预测能力矩阵 ================")
df_res = pd.DataFrame(all_results)
for metric in ['Cls_AUC', 'Recall', 'F1_Score']:
    print(f"\n--- 评估指标: {metric} ---")
    pivot_table = df_res.pivot(index='Crime Type', columns='Model', values=metric)
    print(pivot_table.round(4).to_string())

print("\n================ 数据闭环完成 ================")

In [ ]:
import os
import numpy as np
import pandas as pd
import rasterio
import warnings

warnings.filterwarnings('ignore')

print("================ 1. 从对齐栅格提取数据并还原坐标 ================")
folder_path = r"E:\chuli\NY_aligned"
raster_files = [f for f in os.listdir(folder_path) if f.endswith(('.tif', '.tiff'))]

# 读取坐标系
ref_file = os.path.join(folder_path, raster_files[0])
with rasterio.open(ref_file) as src:
    width, height = src.width, src.height
    transform = src.transform
    cols, rows = np.meshgrid(np.arange(width), np.arange(height))
    xs, ys = rasterio.transform.xy(transform, rows, cols, offset='center')

data_dict = {
    'POINT_X': np.array(xs).flatten(),
    'POINT_Y': np.array(ys).flatten()
}

for file_name in raster_files:
    col_name = os.path.splitext(file_name)[0]
    with rasterio.open(os.path.join(folder_path, file_name)) as src:
        data_dict[col_name] = src.read(1).flatten()

df = pd.DataFrame(data_dict)

print("\n================ 2. 执行空间掩膜与 0 值强制填充 ================")
env_mask_col = 'NY_road_cut_raster_clipped' 

with rasterio.open(os.path.join(folder_path, env_mask_col + '.tif')) as src:
    env_nodata = src.nodata

# 保留纽约有效陆地
if env_nodata is not None:
    df_clean = df[df[env_mask_col] != env_nodata].copy()
else:
    df_clean = df[df[env_mask_col] > -999].copy()

# 将所有未发生犯罪的区域强制赋值为 0
crime_and_year_cols = [col for col in df_clean.columns if 'CrimeType' in col or 'Year_' in col]
for col in crime_and_year_cols:
    with rasterio.open(os.path.join(folder_path, col + '.tif')) as src:
        c_nodata = src.nodata
    if c_nodata is not None:
        df_clean[col] = df_clean[col].replace(c_nodata, 0)
    
    df_clean[col] = np.where(df_clean[col] < 0, 0, df_clean[col])
    df_clean[col] = df_clean[col].fillna(0)

print("🛡️ 安全网格保留完毕。")

print("\n================ 3. 生成 100m 粗栅格并聚合 ================")
GRID_SIZE = 0.001  # 约 100m x 100m
df_clean['coarse_x'] = (df_clean['POINT_X'] // GRID_SIZE) * GRID_SIZE
df_clean['coarse_y'] = (df_clean['POINT_Y'] // GRID_SIZE) * GRID_SIZE

# 配置聚合规则：犯罪次数用 sum，环境指标用 mean
agg_rules = {}
for col in df_clean.columns:
    if col in ['coarse_x', 'coarse_y', 'POINT_X', 'POINT_Y']: continue
    if 'freq' in col.lower() or 'year_' in col.lower() or 'total' in col.lower():
        agg_rules[col] = 'sum'
    else:
        agg_rules[col] = 'mean'

df_coarse = df_clean.groupby(['coarse_x', 'coarse_y']).agg(agg_rules).reset_index()

# 整理坐标列名称，保持整洁
df_coarse.rename(columns={'coarse_x': 'POINT_X', 'coarse_y': 'POINT_Y'}, inplace=True)

print(f"✅ 成功生成 {len(df_coarse)} 个 100m 粗栅格单元。")

print("\n================ 4. 导出 CSV 文件 ================")
output_file = r"E:\chuli\NY_cleaned_dataset_coarse_100m.csv"
df_coarse.to_csv(output_file, index=False)
print(f"💾 大栅格数据已固化并保存至: {output_file}")
print("================ 处理完毕 ================")

In [ ]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

# =========================================================
# 1. 加载数据
# =========================================================
print("================ 1. 加载纽约时空数据 ================")
grid_file = r"E:\chuli\NY_cleaned_dataset_coarse_100m.csv"
arrests_file = r"E:\chuli\NY_dependent_variable\newyork_arrests.csv"

df_grid = pd.read_csv(grid_file)
df_arrests = pd.read_csv(arrests_file)

# 转换时间格式并提取年份
print("🕒 正在解析时间字段...")
df_arrests['ARREST_DAT'] = pd.to_datetime(df_arrests['ARREST_DAT'])
df_arrests['Year'] = df_arrests['ARREST_DAT'].dt.year

# =========================================================
# 2. 空间对齐 (降采样至 100m 栅格)
# =========================================================
print("📍 正在执行空间对齐 (100m 栅格化)...")
GRID_SIZE = 0.001
df_arrests['POINT_X'] = (df_arrests['Longitude'] // GRID_SIZE) * GRID_SIZE
df_arrests['POINT_Y'] = (df_arrests['Latitude'] // GRID_SIZE) * GRID_SIZE

# =========================================================
# 3. 构造历史特征 (2006-2018) 和 未来目标 (2019-2024)
# =========================================================
SPLIT_YEAR = 2018
print(f"✂️ 以 {SPLIT_YEAR} 年为界拆分历史特征与预测目标...")

# A. 提取历史总犯罪频次 (作为强力特征)
history_df = df_arrests[df_arrests['Year'] <= SPLIT_YEAR]
grid_history = history_df.groupby(['POINT_X', 'POINT_Y']).size().reset_index(name='History_Total_Crime')

# B. (进阶) 提取历史分类犯罪频次 (比如历史毒品数预测未来毒品数)
# 这里先做一个总数，如果你需要细分，我们可以再加
print(f"📊 提取到 {len(grid_history)} 个历史热点栅格。")

# =========================================================
# 4. 最终合并 (Merge)
# =========================================================
print("🔄 正在将历史治安基因注入环境大宽表...")

# 将历史犯罪特征 Left Join 到你现有的 100m 栅格表上
# 这里的 df_grid 已经包含了你之前的 [环境特征 + 19-24年犯罪总数]
df_final = pd.merge(df_grid, grid_history, on=['POINT_X', 'POINT_Y'], how='left')

# 填充空值：没发生过历史犯罪的地方填 0
df_final['History_Total_Crime'] = df_final['History_Total_Crime'].fillna(0)

# =========================================================
# 5. 导出终极 CSV
# =========================================================
output_path = r"E:\chuli\NY_final_spatiotemporal_dataset_100m.csv"
df_final.to_csv(output_path, index=False)

print(f"\n================ 🎉 任务完成 ================")
print(f"💾 终极数据集已保存至: {output_path}")
print(f"💡 现在你的特征库增加了 'History_Total_Crime'，快去跑模型吧！")

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score, 
                             recall_score, f1_score, r2_score, mean_absolute_error)
import warnings
warnings.filterwarnings('ignore')

# =========================================================
# 1. 加载终极时空数据集
# =========================================================
print("================ 1. 加载纽约 100m 终极时空数据集 ================")
ny_file = r"E:\chuli\NY_final_spatiotemporal_dataset_100m.csv"
df = pd.read_csv(ny_file)

# 定义特征集：物理环境 + 空间坐标 + 历史犯罪背景
env_features = [col for col in df.columns if (any(k in col for k in ['ratio_', 'accessibility', 'density', 'road', 'landscan'])) and ('arrest' not in col.lower())]
ultimate_features = env_features + ['POINT_X', 'POINT_Y', 'History_Total_Crime']

# 五大犯罪目标列
crime_targets = {
    'Drug and traffic': 'CrimeType_Drug_and_traffic_crimes_Freq',
    'Property': 'CrimeType_Property_crimes_Freq',
    'Public order': 'CrimeType_Public_order_and_other_crimes_Freq',
    'Sexual/Children': 'CrimeType_Sexual_crimes_and_crimes_against_children_Freq',
    'Violent': 'CrimeType_Violent_crimes_Freq'
}

# 模型配置 (Hurdle 架构)
models = {
    "XGBoost": (
        xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, random_state=42, n_jobs=-1, eval_metric='logloss'),
        xgb.XGBRegressor(objective='count:poisson', n_estimators=500, max_depth=7, learning_rate=0.03, random_state=42, n_jobs=-1)
    ),
    "LightGBM": (
        lgb.LGBMClassifier(n_estimators=300, num_leaves=31, learning_rate=0.05, random_state=42, n_jobs=-1, verbose=-1),
        lgb.LGBMRegressor(objective='poisson', n_estimators=600, num_leaves=63, max_depth=8, learning_rate=0.03, random_state=42, n_jobs=-1, verbose=-1)
    ),
    "Random Forest": (
        RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
        RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
    )
}

# =========================================================
# 2. 自动化全量扫描
# =========================================================
print("\n================ 2. 开始执行纽约全犯罪类型时空画像探测 ================")
all_results = []

for crime_name, target_col in crime_targets.items():
    print(f"\n🚀 正在攻坚: 【{crime_name}】 ...")
    
    # 执行 99% 缩尾截断，确保 R2 稳定
    y_raw = df[target_col]
    p99 = np.percentile(y_raw[y_raw > 0], 99)
    y_clipped = np.clip(y_raw, a_min=0, a_max=p99)
    
    X = df[ultimate_features]
    X_train, X_test, y_train_cnt, y_test_cnt = train_test_split(X, y_clipped, test_size=0.2, random_state=42)
    y_train_bin = (y_train_cnt > 0).astype(int)
    y_test_bin = (y_test_cnt > 0).astype(int)
    
    for model_name, (clf, reg) in models.items():
        # 阶段一：分类
        clf.fit(X_train, y_train_bin)
        prob_test = clf.predict_proba(X_test)[:, 1]
        pred_cls = clf.predict(X_test)
        
        auc = roc_auc_score(y_test_bin, prob_test)
        rec = recall_score(y_test_bin, pred_cls, zero_division=0)
        
        # 阶段二：回归
        mask = y_train_cnt > 0
        reg.fit(X_train[mask], y_train_cnt[mask])
        pred_reg = reg.predict(X_test)
        
        # 联合预测
        y_pred_final = prob_test * np.maximum(0, pred_reg)
        r2 = r2_score(y_test_cnt, y_pred_final)
        mae = mean_absolute_error(y_test_cnt, y_pred_final)
        
        all_results.append({
            'Crime Type': crime_name,
            'Model': model_name,
            'Cls_AUC': auc,
            'Cls_Recall': rec,
            'Final_R2': r2,
            'Final_MAE': mae
        })

# =========================================================
# 3. 输出终极全案战报
# =========================================================
print("\n================ 🏆 纽约 100m 时空全模型评估矩阵 ================")
df_res = pd.DataFrame(all_results)

for metric in ['Cls_AUC', 'Cls_Recall', 'Final_R2', 'Final_MAE']:
    print(f"\n--- 评估指标: {metric} ---")
    pivot_table = df_res.pivot(index='Crime Type', columns='Model', values=metric)
    print(pivot_table.round(4).to_string())

print("\n================ 全部闭环！数据现在非常扎实 ================")

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, recall_score, r2_score, mean_absolute_error)
import warnings
warnings.filterwarnings('ignore')

# =========================================================
# 1. 加载终极时空数据集与定义特征组
# =========================================================
print("================ 1. 加载数据与配置消融特征组 ================")
ny_file = r"E:\chuli\NY_final_spatiotemporal_dataset_100m.csv"
df = pd.read_csv(ny_file)

# 基础特征模块
env_features = [col for col in df.columns if (any(k in col for k in ['ratio_', 'accessibility', 'density', 'road', 'landscan'])) and ('arrest' not in col.lower())]
spatial_features = ['POINT_X', 'POINT_Y']
history_feature = ['History_Total_Crime']

# 🌟 定义两大消融实验组
ablation_experiments = {
    "Group 1: 仅环境+坐标 (无历史)": env_features + spatial_features,
    "Group 2: 仅历史+坐标 (无街景)": history_feature + spatial_features
}

crime_targets = {
    'Drug and traffic': 'CrimeType_Drug_and_traffic_crimes_Freq',
    'Property': 'CrimeType_Property_crimes_Freq',
    'Public order': 'CrimeType_Public_order_and_other_crimes_Freq',
    'Sexual/Children': 'CrimeType_Sexual_crimes_and_crimes_against_children_Freq',
    'Violent': 'CrimeType_Violent_crimes_Freq'
}

# 统一模型配置
models = {
    "XGBoost": (
        xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, random_state=42, n_jobs=-1, eval_metric='logloss'),
        xgb.XGBRegressor(objective='count:poisson', n_estimators=500, max_depth=7, learning_rate=0.03, random_state=42, n_jobs=-1)
    ),
    "LightGBM": (
        lgb.LGBMClassifier(n_estimators=300, num_leaves=31, learning_rate=0.05, random_state=42, n_jobs=-1, verbose=-1),
        lgb.LGBMRegressor(objective='poisson', n_estimators=600, num_leaves=63, max_depth=8, learning_rate=0.03, random_state=42, n_jobs=-1, verbose=-1)
    ),
    "Random Forest": (
        RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
        RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
    )
}

# =========================================================
# 2. 执行消融全量扫描
# =========================================================
print("\n================ 2. 开始全矩阵消融实验 (Ablation Study) ================")
all_results = []

for exp_name, feature_set in ablation_experiments.items():
    print(f"\n================ 🛡️ 正在测试: {exp_name} ================")
    
    for crime_name, target_col in crime_targets.items():
        print(f"🚀 攻坚目标: 【{crime_name}】 ...")
        
        # 99% 缩尾处理 (保持与主实验完全一致，确保公平对比)
        y_raw = df[target_col]
        p99 = np.percentile(y_raw[y_raw > 0], 99)
        if p99 > 0:
            y_clipped = np.clip(y_raw, a_min=0, a_max=p99)
        else:
            y_clipped = y_raw
            
        X = df[feature_set]
        X_train, X_test, y_train_cnt, y_test_cnt = train_test_split(X, y_clipped, test_size=0.2, random_state=42)
        
        y_train_bin = (y_train_cnt > 0).astype(int)
        y_test_bin = (y_test_cnt > 0).astype(int)
        
        for model_name, (clf, reg) in models.items():
            # 阶段一：分类
            clf.fit(X_train, y_train_bin)
            prob_test = clf.predict_proba(X_test)[:, 1]
            pred_cls = clf.predict(X_test)
            auc = roc_auc_score(y_test_bin, prob_test) if len(np.unique(y_test_bin)) > 1 else np.nan
            
            # 阶段二：回归
            mask = y_train_cnt > 0
            if mask.sum() > 0:
                reg.fit(X_train[mask], y_train_cnt[mask])
                pred_reg = reg.predict(X_test)
            else:
                pred_reg = np.zeros(len(X_test))
                
            # 联合 Hurdle 预测
            y_pred_final = prob_test * np.maximum(0, pred_reg)
            r2 = r2_score(y_test_cnt, y_pred_final)
            mae = mean_absolute_error(y_test_cnt, y_pred_final)
            
            all_results.append({
                'Experiment': exp_name.split(':')[0], # 简写实验名
                'Crime Type': crime_name,
                'Model': model_name,
                'Cls_AUC': auc,
                'Final_R2': r2,
                'Final_MAE': mae
            })

# =========================================================
# 3. 打印消融对比战报
# =========================================================
print("\n================ 🏆 纽约 100m 消融实验结果对照表 ================")
df_res = pd.DataFrame(all_results)

# 为了方便对比，我们把两组实验的结果并排打印
for metric in ['Cls_AUC', 'Final_R2']:
    print(f"\n--- 核心对比指标: {metric} ---")
    # 按照 Crime Type 和 Model 索引，横向展开两组实验
    pivot_table = df_res.pivot_table(index=['Crime Type', 'Model'], columns='Experiment', values=metric)
    print(pivot_table.round(4).to_string())

print("\n================ 实验彻底闭环！可以开始写 Discussion 了 ================")

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, recall_score, r2_score, mean_absolute_error)
import warnings
warnings.filterwarnings('ignore')

# =========================================================
# 1. 加载终极时空数据集与定义特征组
# =========================================================
print("================ 1. 加载数据与配置消融特征组 ================")
ny_file = r"E:\chuli\NY_final_spatiotemporal_dataset_100m.csv"
df = pd.read_csv(ny_file)

# 基础特征模块
env_features = [col for col in df.columns if (any(k in col for k in ['ratio_', 'accessibility', 'density', 'road', 'landscan'])) and ('arrest' not in col.lower())]
spatial_features = ['POINT_X', 'POINT_Y']
history_feature = ['History_Total_Crime']

# 🌟 定义两大消融实验组
ablation_experiments = {
    "Group 1: 仅环境+坐标 (无历史)": env_features + spatial_features,
    "Group 2: 仅历史+坐标 (无街景)": history_feature + spatial_features
}

crime_targets = {
    'Drug and traffic': 'CrimeType_Drug_and_traffic_crimes_Freq',
    'Property': 'CrimeType_Property_crimes_Freq',
    'Public order': 'CrimeType_Public_order_and_other_crimes_Freq',
    'Sexual/Children': 'CrimeType_Sexual_crimes_and_crimes_against_children_Freq',
    'Violent': 'CrimeType_Violent_crimes_Freq'
}

# 统一模型配置
models = {
    "XGBoost": (
        xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, random_state=42, n_jobs=-1, eval_metric='logloss'),
        xgb.XGBRegressor(objective='count:poisson', n_estimators=500, max_depth=7, learning_rate=0.03, random_state=42, n_jobs=-1)
    ),
    "LightGBM": (
        lgb.LGBMClassifier(n_estimators=300, num_leaves=31, learning_rate=0.05, random_state=42, n_jobs=-1, verbose=-1),
        lgb.LGBMRegressor(objective='poisson', n_estimators=600, num_leaves=63, max_depth=8, learning_rate=0.03, random_state=42, n_jobs=-1, verbose=-1)
    ),
    "Random Forest": (
        RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
        RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
    )
}

# =========================================================
# 2. 执行消融全量扫描
# =========================================================
print("\n================ 2. 开始全矩阵消融实验 (Ablation Study) ================")
all_results = []

for exp_name, feature_set in ablation_experiments.items():
    print(f"\n================ 🛡️ 正在测试: {exp_name} ================")
    
    for crime_name, target_col in crime_targets.items():
        print(f"🚀 攻坚目标: 【{crime_name}】 ...")
        
        # 99% 缩尾处理 (保持与主实验完全一致，确保公平对比)
        y_raw = df[target_col]
        p99 = np.percentile(y_raw[y_raw > 0], 99)
        if p99 > 0:
            y_clipped = np.clip(y_raw, a_min=0, a_max=p99)
        else:
            y_clipped = y_raw
            
        X = df[feature_set]
        X_train, X_test, y_train_cnt, y_test_cnt = train_test_split(X, y_clipped, test_size=0.2, random_state=42)
        
        y_train_bin = (y_train_cnt > 0).astype(int)
        y_test_bin = (y_test_cnt > 0).astype(int)
        
        for model_name, (clf, reg) in models.items():
            # 阶段一：分类
            clf.fit(X_train, y_train_bin)
            prob_test = clf.predict_proba(X_test)[:, 1]
            pred_cls = clf.predict(X_test)
            auc = roc_auc_score(y_test_bin, prob_test) if len(np.unique(y_test_bin)) > 1 else np.nan
            
            # 阶段二：回归
            mask = y_train_cnt > 0
            if mask.sum() > 0:
                reg.fit(X_train[mask], y_train_cnt[mask])
                pred_reg = reg.predict(X_test)
            else:
                pred_reg = np.zeros(len(X_test))
                
            # 联合 Hurdle 预测
            y_pred_final = prob_test * np.maximum(0, pred_reg)
            r2 = r2_score(y_test_cnt, y_pred_final)
            mae = mean_absolute_error(y_test_cnt, y_pred_final)
            
            all_results.append({
                'Experiment': exp_name.split(':')[0], # 简写实验名
                'Crime Type': crime_name,
                'Model': model_name,
                'Cls_AUC': auc,
                'Final_R2': r2,
                'Final_MAE': mae
            })

# =========================================================
# 3. 打印消融对比战报
# =========================================================
print("\n================ 🏆 纽约 100m 消融实验结果对照表 ================")
df_res = pd.DataFrame(all_results)

# 为了方便对比，我们把两组实验的结果并排打印
for metric in ['Cls_AUC', 'Final_R2']:
    print(f"\n--- 核心对比指标: {metric} ---")
    # 按照 Crime Type 和 Model 索引，横向展开两组实验
    pivot_table = df_res.pivot_table(index=['Crime Type', 'Model'], columns='Experiment', values=metric)
    print(pivot_table.round(4).to_string())

print("\n================ 实验彻底闭环！可以开始写 Discussion 了 ================")

In [ ]:
import os
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, recall_score, r2_score, mean_absolute_error
import shap
import matplotlib
matplotlib.use('Agg') # 开启无头模式，防止生成图片时卡死 Notebook
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# =========================================================
# 1. 初始化设置与桌面文件夹创建
# =========================================================
# 自动获取当前用户的桌面路径，并创建 NYpic 文件夹
desktop_path = os.path.join(os.path.expanduser("~"), "Desktop")
nypic_dir = os.path.join(desktop_path, "NYpic")
os.makedirs(nypic_dir, exist_ok=True)
print(f"📁 图像保存目录已就绪: {nypic_dir}")

print("================ 1. 加载终极时空数据集与配置特征组 ================")
ny_file = r"E:\chuli\NY_final_spatiotemporal_dataset_100m.csv"
df = pd.read_csv(ny_file)

# 基础特征模块
env_features = [col for col in df.columns if (any(k in col for k in ['ratio_', 'accessibility', 'density', 'road', 'landscan'])) and ('arrest' not in col.lower())]
spatial_features = ['POINT_X', 'POINT_Y']
history_feature = ['History_Total_Crime']

# 🌟 定义三大终极实验组 (全量 + 两大消融)
experiments = {
    "Full Model (全量时空环境)": env_features + spatial_features + history_feature,
    "Ablation 1 (仅环境+坐标)": env_features + spatial_features,
    "Ablation 2 (仅历史+坐标)": history_feature + spatial_features
}

crime_targets = {
    'Drug and traffic': 'CrimeType_Drug_and_traffic_crimes_Freq',
    'Property': 'CrimeType_Property_crimes_Freq',
    'Public order': 'CrimeType_Public_order_and_other_crimes_Freq',
    'Sexual/Children': 'CrimeType_Sexual_crimes_and_crimes_against_children_Freq',
    'Violent': 'CrimeType_Violent_crimes_Freq'
}

# 统一模型配置
models = {
    "XGBoost": (
        xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, random_state=42, n_jobs=-1, eval_metric='logloss'),
        xgb.XGBRegressor(objective='count:poisson', n_estimators=500, max_depth=7, learning_rate=0.03, random_state=42, n_jobs=-1)
    ),
    "LightGBM": (
        lgb.LGBMClassifier(n_estimators=300, num_leaves=31, learning_rate=0.05, random_state=42, n_jobs=-1, verbose=-1),
        lgb.LGBMRegressor(objective='poisson', n_estimators=600, num_leaves=63, max_depth=8, learning_rate=0.03, random_state=42, n_jobs=-1, verbose=-1)
    ),
    "Random Forest": (
        RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
        RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
    )
}

# =========================================================
# 2. 执行全量与消融评估，并生成 SHAP 图像
# =========================================================
print("\n================ 2. 开始执行全矩阵大拉练与 SHAP 解释 ================")
all_results = []

for exp_name, feature_set in experiments.items():
    print(f"\n================ 🛡️ 正在测试实验组: {exp_name} ================")
    
    for crime_name, target_col in crime_targets.items():
        # 99% 缩尾处理
        y_raw = df[target_col]
        p99 = np.percentile(y_raw[y_raw > 0], 99)
        y_clipped = np.clip(y_raw, a_min=0, a_max=p99) if p99 > 0 else y_raw
            
        X = df[feature_set]
        X_train, X_test, y_train_cnt, y_test_cnt = train_test_split(X, y_clipped, test_size=0.2, random_state=42)
        
        y_train_bin = (y_train_cnt > 0).astype(int)
        y_test_bin = (y_test_cnt > 0).astype(int)
        
        for model_name, (clf, reg) in models.items():
            # ---- 训练阶段一：分类 ----
            clf.fit(X_train, y_train_bin)
            prob_test = clf.predict_proba(X_test)[:, 1]
            auc = roc_auc_score(y_test_bin, prob_test) if len(np.unique(y_test_bin)) > 1 else np.nan
            
            # ---- 训练阶段二：回归 ----
            mask = y_train_cnt > 0
            if mask.sum() > 0:
                reg.fit(X_train[mask], y_train_cnt[mask])
                pred_reg = reg.predict(X_test)
            else:
                pred_reg = np.zeros(len(X_test))
                
            # 联合预测评估
            y_pred_final = prob_test * np.maximum(0, pred_reg)
            r2 = r2_score(y_test_cnt, y_pred_final)
            
            # ---- 保存评估结果 ----
            # 简化实验名称用于表格展示
            if "Full" in exp_name: exp_short = "Full Model"
            elif "1" in exp_name: exp_short = "Ablation 1 (No History)"
            else: exp_short = "Ablation 2 (No StreetView)"
                
            all_results.append({
                'Experiment': exp_short,
                'Crime Type': crime_name,
                'Model': model_name,
                'Cls_AUC': auc,
                'Final_R2': r2
            })
            
            # =========================================================
            # 🌟 核心作图模块：仅对 Full Model (全量模型) 生成 SHAP 依赖图
            # =========================================================
            if "Full" in exp_name:
                try:
                    # 使用回归器(Regressor)来解释发案频次的特征贡献
                    explainer = shap.TreeExplainer(reg)
                    # 为了加速计算并保持图形清晰，我们在测试集中有犯罪记录的样本上计算SHAP
                    X_explain = X_test[y_test_cnt > 0]
                    
                    if len(X_explain) > 0:
                        shap_values = explainer.shap_values(X_explain)
                        
                        # 确保shap_values是正确的维度 (某些版本XGB/LGBM返回列表)
                        if isinstance(shap_values, list):
                            shap_values = shap_values[0]
                            
                        # 绘制 SHAP Summary Plot
                        plt.figure(figsize=(10, 8))
                        shap.summary_plot(shap_values, X_explain, show=False, max_display=15)
                        
                        # 设置中文字体与标题（避免方块字乱码，使用英文最保险）
                        plt.title(f"SHAP Summary: {model_name} - {crime_name}", fontsize=14, pad=20)
                        plt.tight_layout()
                        
                        # 组合文件名并保存
                        safe_crime_name = crime_name.replace("/", "_") # 防止路径报错
                        img_filename = f"SHAP_{model_name}_{safe_crime_name}.png"
                        img_path = os.path.join(nypic_dir, img_filename)
                        
                        plt.savefig(img_path, dpi=300, bbox_inches='tight')
                        plt.close('all') # 彻底清理画布释放内存
                except Exception as e:
                    print(f"⚠️ 生成 {model_name} - {crime_name} 的 SHAP 图时出错: {e}")

print("\n🎉 所有的 SHAP 解释图已成功生成并保存在桌面的 NYpic 文件夹中！")

# =========================================================
# 3. 打印终极大一统对照战报
# =========================================================
print("\n================ 🏆 纽约 100m 全域时空模型终极汇总表 ================")
df_res = pd.DataFrame(all_results)

# 指定列的顺序，让 Full Model 排在第一列
exp_order = ["Full Model", "Ablation 1 (No History)", "Ablation 2 (No StreetView)"]

for metric in ['Cls_AUC', 'Final_R2']:
    print(f"\n--- 核心指标宏观对比: {metric} ---")
    pivot_table = df_res.pivot_table(index=['Crime Type', 'Model'], columns='Experiment', values=metric)
    # 按照指定顺序重新排列列名
    pivot_table = pivot_table.reindex(columns=exp_order)
    print(pivot_table.round(4).to_string())

print("\n================ 项目彻底闭环，祝论文发表顺利！ ================")

In [ ]:
import os
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, recall_score, r2_score, mean_absolute_error
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 💡 控制开关：如果之前已经生成了图片，这里设为 False 可以大幅节省运行时间！
GENERATE_SHAP = False  

# =========================================================
# 1. 初始化设置与桌面文件夹创建
# =========================================================
desktop_path = os.path.join(os.path.expanduser("~"), "Desktop")
nypic_dir = os.path.join(desktop_path, "NYpic")
if GENERATE_SHAP:
    os.makedirs(nypic_dir, exist_ok=True)

print("================ 1. 加载终极时空数据集与配置特征组 ================")
ny_file = r"E:\chuli\NY_final_spatiotemporal_dataset_100m.csv"
df = pd.read_csv(ny_file)

env_features = [col for col in df.columns if (any(k in col for k in ['ratio_', 'accessibility', 'density', 'road', 'landscan'])) and ('arrest' not in col.lower())]
spatial_features = ['POINT_X', 'POINT_Y']
history_feature = ['History_Total_Crime']

experiments = {
    "Full Model (全量时空环境)": env_features + spatial_features + history_feature,
    "Ablation 1 (仅环境+坐标)": env_features + spatial_features,
    "Ablation 2 (仅历史+坐标)": history_feature + spatial_features
}

crime_targets = {
    'Drug and traffic': 'CrimeType_Drug_and_traffic_crimes_Freq',
    'Property': 'CrimeType_Property_crimes_Freq',
    'Public order': 'CrimeType_Public_order_and_other_crimes_Freq',
    'Sexual/Children': 'CrimeType_Sexual_crimes_and_crimes_against_children_Freq',
    'Violent': 'CrimeType_Violent_crimes_Freq'
}

models = {
    "XGBoost": (
        xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, random_state=42, n_jobs=-1, eval_metric='logloss'),
        xgb.XGBRegressor(objective='count:poisson', n_estimators=500, max_depth=7, learning_rate=0.03, random_state=42, n_jobs=-1)
    ),
    "LightGBM": (
        lgb.LGBMClassifier(n_estimators=300, num_leaves=31, learning_rate=0.05, random_state=42, n_jobs=-1, verbose=-1),
        lgb.LGBMRegressor(objective='poisson', n_estimators=600, num_leaves=63, max_depth=8, learning_rate=0.03, random_state=42, n_jobs=-1, verbose=-1)
    ),
    "Random Forest": (
        RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
        RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
    )
}

# =========================================================
# 2. 执行全量与消融评估
# =========================================================
print("\n================ 2. 开始执行四维指标大拉练 ================")
all_results = []

for exp_name, feature_set in experiments.items():
    print(f"\n================ 🛡️ 正在测试实验组: {exp_name} ================")
    
    for crime_name, target_col in crime_targets.items():
        y_raw = df[target_col]
        p99 = np.percentile(y_raw[y_raw > 0], 99)
        y_clipped = np.clip(y_raw, a_min=0, a_max=p99) if p99 > 0 else y_raw
            
        X = df[feature_set]
        X_train, X_test, y_train_cnt, y_test_cnt = train_test_split(X, y_clipped, test_size=0.2, random_state=42)
        
        y_train_bin = (y_train_cnt > 0).astype(int)
        y_test_bin = (y_test_cnt > 0).astype(int)
        
        for model_name, (clf, reg) in models.items():
            # ---- 训练阶段一：分类 ----
            clf.fit(X_train, y_train_bin)
            prob_test = clf.predict_proba(X_test)[:, 1]
            pred_cls = clf.predict(X_test) # 获取 0/1 预测标签用于计算 Recall
            
            auc = roc_auc_score(y_test_bin, prob_test) if len(np.unique(y_test_bin)) > 1 else np.nan
            rec = recall_score(y_test_bin, pred_cls, zero_division=0) # 🌟 新增 Recall
            
            # ---- 训练阶段二：回归 ----
            mask = y_train_cnt > 0
            if mask.sum() > 0:
                reg.fit(X_train[mask], y_train_cnt[mask])
                pred_reg = reg.predict(X_test)
            else:
                pred_reg = np.zeros(len(X_test))
                
            # 联合预测评估
            y_pred_final = prob_test * np.maximum(0, pred_reg)
            r2 = r2_score(y_test_cnt, y_pred_final)
            mae = mean_absolute_error(y_test_cnt, y_pred_final) # 🌟 新增 MAE
            
            # ---- 保存评估结果 ----
            if "Full" in exp_name: exp_short = "Full Model"
            elif "1" in exp_name: exp_short = "Ablation 1 (No History)"
            else: exp_short = "Ablation 2 (No StreetView)"
                
            all_results.append({
                'Experiment': exp_short,
                'Crime Type': crime_name,
                'Model': model_name,
                'Cls_AUC': auc,
                'Cls_Recall': rec, # 🌟 保存 Recall
                'Final_R2': r2,
                'Final_MAE': mae   # 🌟 保存 MAE
            })
            
            # =========================================================
            # SHAP 图像生成模块 (受 GENERATE_SHAP 控制)
            # =========================================================
            if GENERATE_SHAP and "Full" in exp_name:
                try:
                    explainer = shap.TreeExplainer(reg)
                    X_explain = X_test[y_test_cnt > 0]
                    if len(X_explain) > 0:
                        shap_values = explainer.shap_values(X_explain)
                        if isinstance(shap_values, list): shap_values = shap_values[0]
                            
                        plt.figure(figsize=(10, 8))
                        shap.summary_plot(shap_values, X_explain, show=False, max_display=15)
                        plt.title(f"SHAP Summary: {model_name} - {crime_name}", fontsize=14, pad=20)
                        plt.tight_layout()
                        
                        safe_crime_name = crime_name.replace("/", "_")
                        img_path = os.path.join(nypic_dir, f"SHAP_{model_name}_{safe_crime_name}.png")
                        plt.savefig(img_path, dpi=300, bbox_inches='tight')
                        plt.close('all')
                except Exception as e:
                    pass

if GENERATE_SHAP:
    print("\n🎉 所有的 SHAP 解释图已生成！")

# =========================================================
# 3. 打印终极 4 维度大一统对照战报
# =========================================================
print("\n================ 🏆 纽约 100m 全域时空模型终极汇总表 (四维指标) ================")
df_res = pd.DataFrame(all_results)
exp_order = ["Full Model", "Ablation 1 (No History)", "Ablation 2 (No StreetView)"]

# 循环打印 4 个核心指标
for metric in ['Cls_AUC', 'Cls_Recall', 'Final_R2', 'Final_MAE']:
    print(f"\n--- 📍 核心指标宏观对比: {metric} ---")
    pivot_table = df_res.pivot_table(index=['Crime Type', 'Model'], columns='Experiment', values=metric)
    pivot_table = pivot_table.reindex(columns=exp_order)
    
    # 针对不同指标调整小数位数，让表格更专业
    if metric == 'Final_MAE':
        print(pivot_table.round(2).to_string()) # MAE 保留 2 位小数即可
    else:
        print(pivot_table.round(4).to_string()) # AUC/Recall/R2 保留 4 位小数

print("\n================ 表格输出完毕！请直接复制到 Excel 中美化 ================")

In [ ]:
import os
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import shap
import matplotlib
matplotlib.use('Agg') # 开启无头模式，防止连续画 30 张图卡死 Notebook
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# =========================================================
# 1. 文件夹初始化
# =========================================================
desktop_path = os.path.join(os.path.expanduser("~"), "Desktop")
sandian_dir = os.path.join(desktop_path, "sandian")
os.makedirs(sandian_dir, exist_ok=True)
print(f"📁 图像保存目录已就绪: {sandian_dir}")

# =========================================================
# 2. 数据与特征配置
# =========================================================
print("================ 1. 加载纽约时空数据集 ================")
ny_file = r"E:\chuli\NY_final_spatiotemporal_dataset_100m.csv"
df = pd.read_csv(ny_file)

env_features = [col for col in df.columns if (any(k in col for k in ['ratio_', 'accessibility', 'density', 'road', 'landscan'])) and ('arrest' not in col.lower())]
ultimate_features = env_features + ['POINT_X', 'POINT_Y', 'History_Total_Crime']

crime_targets = {
    'Drug and traffic': 'CrimeType_Drug_and_traffic_crimes_Freq',
    'Property': 'CrimeType_Property_crimes_Freq',
    'Public order': 'CrimeType_Public_order_and_other_crimes_Freq',
    'Sexual_Children': 'CrimeType_Sexual_crimes_and_crimes_against_children_Freq', # 替换了斜杠防报错
    'Violent': 'CrimeType_Violent_crimes_Freq'
}

# 回归器配置（只需回归器来解释频次的非线性特征）
models = {
    "XGBoost": xgb.XGBRegressor(objective='count:poisson', n_estimators=500, max_depth=7, learning_rate=0.03, random_state=42, n_jobs=-1),
    "LightGBM": lgb.LGBMRegressor(objective='poisson', n_estimators=600, num_leaves=63, max_depth=8, learning_rate=0.03, random_state=42, n_jobs=-1, verbose=-1),
    "RandomForest": RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
}

# 一个简单的小函数，用来把冗长的变量名变成好看的图表标题
def clean_feature_name(name):
    clean_name = name.replace('ratio_NY_plus_', '').replace('_clipped', '').replace('_', ' ')
    return clean_name.title()

# =========================================================
# 3. 开始自动化扫荡：3大模型 × 5大犯罪
# =========================================================
print("\n================ 2. 开始全矩阵扫荡与动态提取 Top 2 散点图 ================")

for crime_name, target_col in crime_targets.items():
    print(f"\n🚀 正在攻坚目标: 【{crime_name}】 ...")
    
    # 缩尾处理
    y_raw = df[target_col]
    p99 = np.percentile(y_raw[y_raw > 0], 99)
    y_clipped = np.clip(y_raw, a_min=0, a_max=p99) if p99 > 0 else y_raw
        
    X = df[ultimate_features]
    X_train, X_test, y_train, y_test = train_test_split(X, y_clipped, test_size=0.2, random_state=42)
    
    mask_train = y_train > 0
    mask_test = y_test > 0
    
    # 针对 3 个模型分别训练和解释
    for model_name, reg in models.items():
        if mask_train.sum() == 0:
            continue
            
        # 训练模型
        reg.fit(X_train[mask_train], y_train[mask_train])
        
        # 计算 SHAP
        explainer = shap.TreeExplainer(reg)
        X_explain = X_test[mask_test]
        
        if len(X_explain) == 0:
            continue
            
        shap_values = explainer.shap_values(X_explain)
        if isinstance(shap_values, list):
            shap_values = shap_values[0]
            
        # =========================================================
        # 🌟 核心智能模块：动态计算重要性并提取 Top 2 物理特征
        # =========================================================
        # 计算每个特征的平均绝对 SHAP 值作为重要性
        mean_abs_shap = np.abs(shap_values).mean(axis=0)
        feat_importance = pd.DataFrame({
            'feature': X_explain.columns,
            'importance': mean_abs_shap
        })
        
        # 强行剔除历史数据和坐标，只保留纯物理环境特征
        exclude_feats = ['History_Total_Crime', 'POINT_X', 'POINT_Y']
        env_importance = feat_importance[~feat_importance['feature'].isin(exclude_feats)]
        
        # 获取排名前 2 的物理特征名称
        top_2_features = env_importance.sort_values(by='importance', ascending=False).head(2)['feature'].tolist()
        
        # =========================================================
        # 画出这 2 个特征的散点图
        # =========================================================
        for rank, feat in enumerate(top_2_features):
            try:
                plt.figure(figsize=(8, 6))
                
                # 绘制 SHAP 依赖散点图，交互特征统一使用历史治安背景
                shap.dependence_plot(
                    feat, 
                    shap_values, 
                    X_explain, 
                    interaction_index='History_Total_Crime',
                    show=False,
                    cmap=plt.get_cmap("cool") 
                )
                
                pretty_name = clean_feature_name(feat)
                plt.title(f"{model_name} - {crime_name}\nTop {rank+1} Env Feature: {pretty_name}", fontsize=14, pad=15)
                plt.tight_layout()
                
                # 组合极其清晰的文件名，方便你后续在文件夹里查找
                safe_crime = crime_name.replace(" ", "_")
                img_name = f"Scatter_{model_name}_{safe_crime}_Top{rank+1}_{pretty_name.replace(' ', '')}.png"
                img_path = os.path.join(sandian_dir, img_name)
                
                plt.savefig(img_path, dpi=300, bbox_inches='tight')
                plt.close('all')
            except Exception as e:
                print(f"⚠️ 生成 {model_name} - {crime_name} - {feat} 时出错: {e}")
                plt.close('all')

print(f"\n================ 🎉 任务彻底完成！ ================")
print(f"请直接前往桌面文件夹 【sandian】 验收你的 30 张核心微观机制散点图！")

In [ ]:
import os
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
import shap
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# =========================================================
# 1. 文件夹与数据加载
# =========================================================
desktop_path = os.path.join(os.path.expanduser("~"), "Desktop")
print("================ Loading NY Spatiotemporal Dataset ================")
ny_file = r"E:\chuli\NY_final_spatiotemporal_dataset_100m.csv"
df = pd.read_csv(ny_file)

# 获取特征列
env_features = [col for col in df.columns if (any(k in col for k in ['ratio_', 'accessibility', 'density', 'road', 'landscan'])) and ('arrest' not in col.lower())]
ultimate_features = env_features + ['POINT_X', 'POINT_Y', 'History_Total_Crime'] 

target_col = 'CrimeType_Violent_crimes_Freq' # 以暴力犯罪为例

# =========================================================
# 2. 纯英文学术名称映射字典 (Pure English Mapping)
# =========================================================
# ⚠️ 注意：请将 'Your_Night_Pop_Column_Name' 替换为你数据集中真实的人口列名！
# 我在这里写了几个常见的 landscan 变量名作为占位符，如果能匹配上会自动替换。
feature_name_mapping = {
    'landscan-usa-2021-conus-night_ny_clipped_clipped': 'Nighttime Population',  # 请替换为你的实际夜间人口列名
    'landscan-usa-2021-conus-day_ny_clipped_clipped': 'Daytime Population',      # 请替换为你的实际日间人口列名
    
    # 街景与建成环境特征纯英文映射
    'ratio_NY_plus_Enclosure_degree_clipped': 'Enclosure Degree',
    'ratio_NY_Enclosure_degree_clipped': 'Enclosure Degree',
    'ratio_NY_plus_Pedestrian_density_clipped': 'Pedestrian Density',
    'ratio_NY_Pedestrian_density_clipped': 'Pedestrian Density',
    'ratio_NY_plus_Traffic_flow_clipped': 'Traffic Flow',
    'ratio_NY_Traffic_flow_clipped': 'Traffic Flow',
    'ratio_NY_plus_Greening_degree_clipped': 'Greening Degree',
    'ratio_NY_Greening_degree_clipped': 'Greening Degree',
    'NY_plus_school_to_shop_accessibility_10m_clipped': 'Shop Accessibility',
    'NY_school_to_shop_accessibility_10m_clipped': 'Shop Accessibility',
    'NY_plus_school_bus_accessibility_clipped': 'School Bus Accessibility',
    'NY_school_bus_accessibility_clipped': 'School Bus Accessibility',
    'ratio_NY_plus_Openness_clipped': 'Sky Openness',
    'ratio_NY_Openness_clipped': 'Sky Openness',
    'ratio_NY_plus_Street_furniture_clipped': 'Street Furniture',
    'ratio_NY_Street_furniture_clipped': 'Street Furniture',
    'ratio_NY_plus_Surveillance_and_lighting_clipped': 'Lighting & Surveillance',
    'ratio_NY_Surveillance_and_lighting_clipped': 'Lighting & Surveillance'
}

def get_academic_name(col_name):
    for key, val in feature_name_mapping.items():
        if key in col_name or col_name in key:
            return val
    # Fallback cleaning if not in dictionary
    clean_name = col_name.replace('ratio_NY_plus_', '').replace('ratio_NY_', '').replace('NY_plus_', '')
    clean_name = clean_name.replace('_clipped', '').replace('_', ' ')
    return clean_name.title()

# =========================================================
# 3. 训练模型与计算 SHAP (XGBoost)
# =========================================================
print("🚀 Training XGBoost and calculating SHAP values...")
y_raw = df[target_col]
p99 = np.percentile(y_raw[y_raw > 0], 99)
y_clipped = np.clip(y_raw, a_min=0, a_max=p99) if p99 > 0 else y_raw
    
X = df[ultimate_features]
X_train, X_test, y_train, y_test = train_test_split(X, y_clipped, test_size=0.2, random_state=42)

mask_train = y_train > 0
mask_test = y_test > 0

reg = xgb.XGBRegressor(objective='count:poisson', n_estimators=500, max_depth=7, learning_rate=0.03, random_state=42, n_jobs=-1)
reg.fit(X_train[mask_train], y_train[mask_train])

explainer = shap.TreeExplainer(reg)
X_explain = X_test[mask_test]
shap_values = explainer.shap_values(X_explain)

# =========================================================
# 4. 【核心优化】一网打尽：剔除历史基线 + XY坐标
# =========================================================
# 定义要剥离的“霸屏”变量名单
blacklist_features = ['History_Total_Crime', 'POINT_X', 'POINT_Y']

# 获取黑名单特征在矩阵中的索引
cols_to_drop_idx = [X_explain.columns.get_loc(col) for col in blacklist_features if col in X_explain.columns]

if cols_to_drop_idx:
    # 从 SHAP numpy 数组中剔除这些列
    shap_values_filtered = np.delete(shap_values, cols_to_drop_idx, axis=1)
    # 从 DataFrame 中剔除这些列
    X_explain_filtered = X_explain.drop(columns=[col for col in blacklist_features if col in X_explain.columns])
    print(f"✅ Successfully removed dominant/spatial features: {blacklist_features}")
else:
    shap_values_filtered = shap_values
    X_explain_filtered = X_explain.copy()

# 重命名为纯英文高级词汇
X_explain_renamed = X_explain_filtered.rename(columns=lambda x: get_academic_name(x))

# =========================================================
# 5. 绘制并保存最终蜂群图 (Pure English)
# =========================================================
# 强制使用英文字体
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']

plt.figure(figsize=(10, 6))

shap.summary_plot(
    shap_values_filtered, 
    X_explain_renamed, 
    max_display=10,  # 显示 Top 10
    show=False       
)

# 调整边界防止左侧文字被截断
plt.gcf().set_size_inches(10.5, 6.5)
plt.tight_layout()

# 保存至桌面
output_path = os.path.join(desktop_path, "Figure_SHAP_NY_Violent_Crime.png")
plt.savefig(output_path, dpi=300, bbox_inches='tight')

print(f"\n🎉 Done! High-res English plot saved to Desktop: {output_path}")
plt.show()

In [ ]:
import os
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
import shap
import matplotlib
matplotlib.use('Agg') # 开启无头模式
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# =========================================================
# 1. 文件夹与数据配置
# =========================================================
desktop_path = os.path.join(os.path.expanduser("~"), "Desktop")
output_dir = os.path.join(desktop_path, "NY_暴力犯罪_微观特征散点图_Top4")
os.makedirs(output_dir, exist_ok=True)
print(f"📁 图像保存目录已就绪: {output_dir}")

print("================ 1. 加载纽约时空数据集 ================")
ny_file = r"E:\chuli\NY_final_spatiotemporal_dataset_100m.csv"
df = pd.read_csv(ny_file)

env_features = [col for col in df.columns if (any(k in col for k in ['ratio_', 'accessibility', 'density', 'road', 'landscan'])) and ('arrest' not in col.lower())]
ultimate_features = env_features + ['POINT_X', 'POINT_Y', 'History_Total_Crime']
target_col = 'CrimeType_Violent_crimes_Freq' # 依然以暴力犯罪为例

# =========================================================
# 2. 纯英文学术名称映射字典
# =========================================================
feature_name_mapping = {
    'ratio_NY_plus_Enclosure_degree_clipped': 'Enclosure Degree',
    'ratio_NY_Enclosure_degree_clipped': 'Enclosure Degree',
    'ratio_NY_plus_Pedestrian_density_clipped': 'Pedestrian Density',
    'ratio_NY_Pedestrian_density_clipped': 'Pedestrian Density',
    'ratio_NY_plus_Traffic_flow_clipped': 'Traffic Flow',
    'ratio_NY_Traffic_flow_clipped': 'Traffic Flow',
    'ratio_NY_plus_Greening_degree_clipped': 'Greening Degree',
    'ratio_NY_Greening_degree_clipped': 'Greening Degree',
    'NY_plus_school_to_shop_accessibility_10m_clipped': 'Shop Accessibility',
    'NY_school_to_shop_accessibility_10m_clipped': 'Shop Accessibility',
    'NY_plus_school_bus_accessibility_clipped': 'School Bus Accessibility',
    'NY_school_bus_accessibility_clipped': 'School Bus Accessibility',
    'ratio_NY_plus_Openness_clipped': 'Sky Openness',
    'ratio_NY_Openness_clipped': 'Sky Openness',
    'ratio_NY_plus_Street_furniture_clipped': 'Street Furniture',
    'ratio_NY_Street_furniture_clipped': 'Street Furniture',
    'ratio_NY_plus_Surveillance_and_lighting_clipped': 'Lighting & Surveillance',
    'ratio_NY_Surveillance_and_lighting_clipped': 'Lighting & Surveillance',
    'History_Total_Crime': 'Historical Crime Level'
}

def get_academic_name(col_name):
    for key, val in feature_name_mapping.items():
        if key in col_name or col_name in key:
            return val
    clean_name = col_name.replace('ratio_NY_plus_', '').replace('ratio_NY_', '').replace('NY_plus_', '')
    clean_name = clean_name.replace('_clipped', '').replace('_', ' ')
    return clean_name.title()

# =========================================================
# 3. 训练 XGBoost 与计算 SHAP
# =========================================================
print("\n🚀 正在训练 XGBoost 并计算 SHAP...")
y_raw = df[target_col]
p99 = np.percentile(y_raw[y_raw > 0], 99)
y_clipped = np.clip(y_raw, a_min=0, a_max=p99) if p99 > 0 else y_raw
    
X = df[ultimate_features]
X_train, X_test, y_train, y_test = train_test_split(X, y_clipped, test_size=0.2, random_state=42)

mask_train = y_train > 0
mask_test = y_test > 0

reg = xgb.XGBRegressor(objective='count:poisson', n_estimators=500, max_depth=7, learning_rate=0.03, random_state=42, n_jobs=-1)
reg.fit(X_train[mask_train], y_train[mask_train])

explainer = shap.TreeExplainer(reg)
X_explain = X_test[mask_test]
shap_values = explainer.shap_values(X_explain)

# =========================================================
# 4. 🌟 强制过滤：只抓取微观物理特征 Top 4
# =========================================================
mean_abs_shap = np.abs(shap_values).mean(axis=0)
feat_importance = pd.DataFrame({
    'feature': X_explain.columns,
    'importance': mean_abs_shap
})

# 定义黑名单：历史、坐标、人口（拦截包含 landscan 或 pop 的特征）
black_list_keywords = ['History', 'POINT', 'landscan', 'pop', 'night', 'day']

# 过滤出“干净”的物理建成环境特征
pure_env_features = feat_importance[
    ~feat_importance['feature'].str.contains('|'.join(black_list_keywords), case=False)
]

# 提取排名前 4 的微观特征
top_4_features = pure_env_features.sort_values(by='importance', ascending=False).head(4)['feature'].tolist()

# 翻译所有列名为英文学术词汇
X_explain_renamed = X_explain.rename(columns=lambda x: get_academic_name(x))
interaction_feat_academic = get_academic_name('History_Total_Crime')

# =========================================================
# 5. 绘制并保存 4 张散点图
# =========================================================
# 强制英文字体
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']

for rank, raw_feat_name in enumerate(top_4_features):
    try:
        plt.figure(figsize=(8, 6))
        feat_academic = get_academic_name(raw_feat_name)
        
        # 画图：将交互特征设为历史犯罪率，用以展示颜色深度
        shap.dependence_plot(
            feat_academic, 
            shap_values, 
            X_explain_renamed, 
            interaction_index=interaction_feat_academic,
            show=False,
            cmap=plt.get_cmap("cool") 
        )
        
        # 纯英文标题，去掉代码痕迹
        plt.title(f"XGBoost - Violent Crime\nTop {rank+1} Micro-Env Driver: {feat_academic}", fontsize=14, pad=15, fontweight='bold')
        plt.tight_layout()
        
        # 保存图片
        img_name = f"Figure_Scatter_NY_Violent_Top{rank+1}_{feat_academic.replace(' ', '')}.png"
        img_path = os.path.join(output_dir, img_name)
        plt.savefig(img_path, dpi=300, bbox_inches='tight')
        plt.close('all')
        
    except Exception as e:
        print(f"⚠️ 生成 {raw_feat_name} 时出错: {e}")
        plt.close('all')

print(f"\n================ 🎉 任务完成！ ================")
print(f"请前往桌面文件夹 【NY_暴力犯罪_微观特征散点图_Top4】 查看 4 张纯正的微观机制散点图。")
print(f"提取出的特征依次是: {[get_academic_name(f) for f in top_4_features]}")

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image

desktop_path = os.path.join(os.path.expanduser("~"), "Desktop")
# 👉 新增：指向存放那 4 张图片的专属文件夹
img_folder = os.path.join(desktop_path, "NY_暴力犯罪_微观特征散点图_Top4")

# 更新路径：去 img_folder 里面找图片
images = [
    os.path.join(img_folder, "Figure_Scatter_NY_Violent_Top3_EnclosureDegree.png"),
    os.path.join(img_folder, "Figure_Scatter_NY_Violent_Top1_StreetFurniture.png"),
    os.path.join(img_folder, "Figure_Scatter_NY_Violent_Top2_TrafficFlow.png"),
    os.path.join(img_folder, "Figure_Scatter_NY_Violent_Top4_ShopAccessibility.png")
]
labels = ['(a)', '(b)', '(c)', '(d)']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, img_path, label in zip(axes.flatten(), images, labels):
    try:
        img = Image.open(img_path)
        ax.imshow(img)
        ax.axis('off')
        # 在左上角打上学术角标 (a), (b), (c), (d)
        ax.text(0.02, 0.98, label, transform=ax.transAxes, fontsize=24, fontweight='bold', va='top')
    except FileNotFoundError:
        print(f"❌ 找不到图片: {img_path}")
        print("请检查该图片是否真的在这个文件夹里，或者文件名有没有拼写错误。")

plt.tight_layout()
# 拼好的大图会直接保存在你的桌面上
output_fig = os.path.join(desktop_path, "Figure_Y_NY_Micro_Mechanisms_Combined.png")
plt.savefig(output_fig, dpi=300, bbox_inches='tight')
print(f"\n🎉 2x2 拼图已生成至桌面: {output_fig}")